In [ ]:
CREATE OR REPLACE VIEW alme_dev_silver.hierarchy_engine.v_hierarchy_paths AS
WITH RECURSIVE hierarchy_cte AS (

    SELECT
        n.hierarchy_id,
        n.version_id,
        n.account_key,
        n.account_name,
        n.parent_account_key,
        CAST(array(n.account_key) AS array<string>) AS path_keys,
        CAST(array(n.account_name) AS array<string>) AS path_names,
        1 AS depth,
        n.account_key AS root_account_key,
        n.account_name AS root_account_name
    FROM alme_dev_silver.hierarchy_engine.base_hierarchy_node n
    WHERE n.parent_account_key IS NULL

    UNION ALL

    SELECT
        c.hierarchy_id,
        c.version_id,
        c.account_key,
        c.account_name,
        c.parent_account_key,
        concat(p.path_keys, array(c.account_key)) AS path_keys,
        concat(p.path_names, array(c.account_name)) AS path_names,
        p.depth + 1 AS depth,
        p.root_account_key,
        p.root_account_name
    FROM alme_dev_silver.hierarchy_engine.base_hierarchy_node c
    JOIN hierarchy_cte p
      ON c.hierarchy_id = p.hierarchy_id
     AND c.version_id = p.version_id
     AND c.parent_account_key = p.account_key
    WHERE NOT array_contains(p.path_keys, c.account_key)
)
SELECT
    hierarchy_id,
    version_id,
    account_key,
    account_name,
    parent_account_key,
    path_keys,
    path_names,
    depth,
    root_account_key,
    root_account_name
FROM hierarchy_cte;


# notebooks/02_rebuild_hierarchy_views.py

"""
Rebuild flattened hierarchy and dimension views from base tables.

This notebook should be run after:
1. publishing a hierarchy YAML to base tables
2. updating hierarchy versions
"""

row = spark.sql("""
    SELECT MAX(depth) AS max_depth
    FROM alme_dev_silver.hierarchy_engine.v_hierarchy_paths
""").first()

max_depth = int(row.max_depth) if row and row.max_depth is not None else 0
if max_depth == 0:
    raise ValueError("No hierarchy depth found in v_hierarchy_paths")

select_cols = [
    "p.hierarchy_id",
    "p.version_id",
    "p.account_key",
    "p.account_name",
    "p.parent_account_key",
    "p.depth",
    "p.root_account_key",
    "p.root_account_name",
    "p.path_keys",
    "p.path_names",
    "child.account_key IS NULL AS derived_is_leaf"
]

for idx in range(max_depth):
    level_num = idx + 1

    key_expr = f"""
        CASE
            WHEN p.depth > {idx} THEN get(p.path_keys, {idx})
            ELSE element_at(p.path_keys, -1)
        END
    """

    name_expr = f"""
        CASE
            WHEN p.depth > {idx} THEN get(p.path_names, {idx})
            ELSE element_at(p.path_names, -1)
        END
    """

    select_cols.append(f"{key_expr} AS level{level_num}_key")
    select_cols.append(f"{name_expr} AS level{level_num}_name")
    select_cols.append(f"CAST({key_expr} AS INT) AS level{level_num}_sort")

select_clause = ",\n    ".join(select_cols)

sql_stmt = f"""
CREATE OR REPLACE VIEW alme_dev_silver.hierarchy_engine.v_hierarchy_flat AS
SELECT
    {select_clause}
FROM alme_dev_silver.hierarchy_engine.v_hierarchy_paths p
LEFT JOIN alme_dev_silver.hierarchy_engine.base_hierarchy_node child
  ON p.hierarchy_id = child.hierarchy_id
 AND p.version_id = child.version_id
 AND p.account_key = child.parent_account_key
"""

spark.sql(sql_stmt)
print("Created v_hierarchy_flat")





In [ ]:
# notebooks/02_rebuild_hierarchy_views.py

row = spark.sql("""
    SELECT MAX(depth) AS max_depth
    FROM alme_dev_silver.hierarchy_engine.v_hierarchy_flat
""").first()

max_depth = int(row.max_depth) if row and row.max_depth is not None else 0
if max_depth == 0:
    raise ValueError("No hierarchy depth found in v_hierarchy_flat")

select_cols = [
    "f.hierarchy_id",
    "r.hierarchy_name",
    "r.hierarchy_description",
    "r.owner_team",
    "r.business_domain",
    "f.version_id",
    "v.version_name",
    "v.version_status",
    "v.effective_start_date",
    "v.effective_end_date",
    "v.is_current",
    "f.account_key AS leaf_key",
    "f.account_name AS leaf_name",
    "concat(f.hierarchy_id, '||', f.account_key) AS hier_leaf_key",
    "concat(f.hierarchy_id, '||', f.version_id, '||', f.account_key) AS hier_ver_leaf_key",
    "f.depth",
    "f.root_account_key",
    "f.root_account_name"
]

for idx in range(max_depth):
    level_num = idx + 1
    select_cols.append(f"f.level{level_num}_key")
    select_cols.append(f"f.level{level_num}_name")
    select_cols.append(f"f.level{level_num}_sort")

select_clause = ",\n    ".join(select_cols)

sql_stmt = f"""
CREATE OR REPLACE VIEW alme_dev_silver.hierarchy_engine.v_hierarchy_dims AS
SELECT
    {select_clause}
FROM alme_dev_silver.hierarchy_engine.v_hierarchy_flat f
JOIN alme_dev_silver.hierarchy_engine.hierarchy_registry r
  ON f.hierarchy_id = r.hierarchy_id
JOIN alme_dev_silver.hierarchy_engine.hierarchy_version v
  ON f.hierarchy_id = v.hierarchy_id
 AND f.version_id = v.version_id
WHERE f.derived_is_leaf = TRUE
"""

spark.sql(sql_stmt)
print("Created v_hierarchy_dims")


In [ ]:
# notebooks/02_rebuild_hierarchy_views.py

row = spark.sql("""
    SELECT MAX(depth) AS max_depth
    FROM alme_dev_silver.hierarchy_engine.v_hierarchy_dims
""").first()

max_depth = int(row.max_depth) if row and row.max_depth is not None else 0
if max_depth == 0:
    raise ValueError("No hierarchy depth found in v_hierarchy_dims")

publish_cols = [
    "hierarchy_id",
    "hierarchy_name",
    "hierarchy_description",
    "owner_team",
    "business_domain",
    "version_id",
    "version_name",
    "version_status",
    "effective_start_date",
    "effective_end_date",
    "is_current",
    "leaf_key",
    "leaf_name",
    "hier_leaf_key",
    "hier_ver_leaf_key",
    "depth",
    "root_account_key",
    "root_account_name"
]

for idx in range(max_depth):
    level_num = idx + 1
    publish_cols.extend([
        f"level{level_num}_key",
        f"level{level_num}_name",
        f"level{level_num}_sort",
    ])

publish_clause = ",\n    ".join(publish_cols)

sql_stmt = f"""
CREATE OR REPLACE VIEW alme_dev_silver.hierarchy_engine.dim_reporting_hierarchy AS
SELECT
    {publish_clause}
FROM alme_dev_silver.hierarchy_engine.v_hierarchy_dims
WHERE version_status = 'published'
"""

spark.sql(sql_stmt)
print("Created dim_reporting_hierarchy")